# 05 · Triton kernels: vector add, fused softmax, tiled matmul, benchmarking

**Needed for:** FlashAttention-2 (LLM track, step 3), int4 matmul (step 4), sparse attention kernels (step 9).

Triton programs describe what **one block** of work does. You launch a grid of program instances; each one
loads a tile with `tl.load`, computes on it, and stores with `tl.store`. Triton handles the per-thread details
(coalescing, shared memory, tensor cores) that CUDA makes you write by hand.

Triton also compiles for AMD GPUs through ROCm, so the same kernels run on MI300X.

**Requires a GPU.** On Colab choose L4 or A100 (T4 lacks bf16). Every cell skips cleanly on CPU.

In [ ]:
import importlib.util
import torch

torch.manual_seed(0)
HAS_GPU = torch.cuda.is_available() and importlib.util.find_spec("triton") is not None
if HAS_GPU:
    import triton
    import triton.language as tl
    print("GPU:", torch.cuda.get_device_name(), "| triton", triton.__version__)
else:
    print("No GPU/triton: the kernel cells will be skipped")


def drill(test, fn):
    """Run a test against your own implementation; prints TODO until you write it."""
    try:
        test(fn)
    except NotImplementedError:
        print(f"TODO  {fn.__name__}")
    else:
        print(f"PASS  {fn.__name__}")

## 1. Vector add: the skeleton of every kernel

1. `pid = tl.program_id(0)`: which block am I?
2. `offsets = pid * BLOCK + tl.arange(0, BLOCK)`: which elements does this block own?
3. `mask = offsets < n`: the last block may run past the end.
4. Load, compute, store.

`BLOCK` is a `tl.constexpr`: known at compile time, and must be a power of two.

In [ ]:
if HAS_GPU:
    @triton.jit
    def add_kernel(x_ptr, y_ptr, out_ptr, n, BLOCK: tl.constexpr):
        pid = tl.program_id(0)
        offsets = pid * BLOCK + tl.arange(0, BLOCK)
        mask = offsets < n
        x = tl.load(x_ptr + offsets, mask=mask)
        y = tl.load(y_ptr + offsets, mask=mask)
        tl.store(out_ptr + offsets, x + y, mask=mask)

    def triton_add(x, y):
        out = torch.empty_like(x)
        n = x.numel()
        grid = (triton.cdiv(n, 1024),)
        add_kernel[grid](x, y, out, n, BLOCK=1024)
        return out


def test_add(fn):
    for n in (1, 1000, 98_765):  # not multiples of the block size
        x, y = torch.randn(n, device="cuda"), torch.randn(n, device="cuda")
        torch.testing.assert_close(fn(x, y), x + y)


if HAS_GPU:
    test_add(triton_add)
    print("vector add ok")

## 2. Fused row softmax

One program per row. The whole row fits in one block (`BLOCK` = next power of two ≥ number of columns), so the
row is read from HBM once and written once. PyTorch's unfused version reads it several times (max, subtract,
exp, sum, divide).

Masked-out lanes load `-inf`, so they contribute `exp(-inf) = 0` to the sum.

In [ ]:
if HAS_GPU:
    @triton.jit
    def softmax_kernel(out_ptr, in_ptr, in_row_stride, out_row_stride, n_cols, BLOCK: tl.constexpr):
        row = tl.program_id(0)
        cols = tl.arange(0, BLOCK)
        mask = cols < n_cols
        x = tl.load(in_ptr + row * in_row_stride + cols, mask=mask, other=-float("inf")).to(tl.float32)
        x = x - tl.max(x, axis=0)
        num = tl.exp(x)
        out = num / tl.sum(num, axis=0)
        tl.store(out_ptr + row * out_row_stride + cols, out, mask=mask)

    def triton_softmax(x):
        n_rows, n_cols = x.shape
        out = torch.empty_like(x)
        softmax_kernel[(n_rows,)](out, x, x.stride(0), out.stride(0), n_cols,
                                  BLOCK=triton.next_power_of_2(n_cols))
        return out


def test_softmax(fn):
    for shape in ((4, 7), (128, 1000), (3, 4096)):
        x = torch.randn(shape, device="cuda") * 5
        torch.testing.assert_close(fn(x), torch.softmax(x, -1))


if HAS_GPU:
    test_softmax(triton_softmax)
    print("softmax ok")

## 3. Tiled matmul with `tl.dot`

Each program computes one `BLOCK_M × BLOCK_N` tile of C, walking along K in steps of `BLOCK_K`.
Pointer arithmetic uses strides, so the kernel works on transposed or sliced inputs too.
`tl.dot` maps to tensor cores; it needs tile sizes of at least 16. Accumulate in fp32, cast when storing.

In [ ]:
if HAS_GPU:
    @triton.jit
    def matmul_kernel(a_ptr, b_ptr, c_ptr, M, N, K,
                      stride_am, stride_ak, stride_bk, stride_bn, stride_cm, stride_cn,
                      BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr, BLOCK_K: tl.constexpr):
        pid_m, pid_n = tl.program_id(0), tl.program_id(1)
        offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
        offs_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
        offs_k = tl.arange(0, BLOCK_K)
        a_ptrs = a_ptr + offs_m[:, None] * stride_am + offs_k[None, :] * stride_ak
        b_ptrs = b_ptr + offs_k[:, None] * stride_bk + offs_n[None, :] * stride_bn
        acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)
        for k0 in range(0, K, BLOCK_K):
            a = tl.load(a_ptrs, mask=(offs_m[:, None] < M) & (offs_k[None, :] + k0 < K), other=0.0)
            b = tl.load(b_ptrs, mask=(offs_k[:, None] + k0 < K) & (offs_n[None, :] < N), other=0.0)
            acc += tl.dot(a, b)
            a_ptrs += BLOCK_K * stride_ak
            b_ptrs += BLOCK_K * stride_bk
        c_ptrs = c_ptr + offs_m[:, None] * stride_cm + offs_n[None, :] * stride_cn
        tl.store(c_ptrs, acc.to(c_ptr.dtype.element_ty), mask=(offs_m[:, None] < M) & (offs_n[None, :] < N))

    def triton_matmul(a, b, BLOCK_M=64, BLOCK_N=64, BLOCK_K=32):
        M, K = a.shape
        _, N = b.shape
        c = torch.empty(M, N, device=a.device, dtype=a.dtype)
        grid = (triton.cdiv(M, BLOCK_M), triton.cdiv(N, BLOCK_N))
        matmul_kernel[grid](a, b, c, M, N, K, a.stride(0), a.stride(1), b.stride(0), b.stride(1),
                            c.stride(0), c.stride(1), BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N, BLOCK_K=BLOCK_K)
        return c


def test_matmul(fn):
    for M, K, N in ((64, 64, 64), (100, 70, 130), (512, 256, 384)):
        a = torch.randn(M, K, device="cuda", dtype=torch.float16)
        b = torch.randn(K, N, device="cuda", dtype=torch.float16)
        torch.testing.assert_close(fn(a, b), a @ b, atol=2e-2, rtol=1e-2)
    bt = torch.randn(96, 80, device="cuda", dtype=torch.float16).t()   # non-contiguous input
    a = torch.randn(64, 80, device="cuda", dtype=torch.float16)
    torch.testing.assert_close(fn(a, bt), a @ bt, atol=2e-2, rtol=1e-2)


if HAS_GPU:
    test_matmul(triton_matmul)
    print("matmul ok")

## 4. Benchmarking properly

GPU calls are asynchronous: timing with `time.time()` around a launch measures only the launch.
`triton.testing.do_bench` warms up, synchronizes, flushes the L2 cache between runs, and returns a median in
milliseconds.

For memory-bound kernels, report **effective bandwidth** (bytes read + written ÷ time) and compare to the GPU's
peak. For compute-bound ones, report **TFLOP/s**.

In [ ]:
if HAS_GPU:
    from triton.testing import do_bench

    x = torch.randn(4096, 4096, device="cuda")
    gbps = lambda ms: 2 * x.numel() * x.element_size() / (ms * 1e-3) / 1e9   # one read + one write
    ms_torch = do_bench(lambda: torch.softmax(x, -1))
    ms_triton = do_bench(lambda: triton_softmax(x))
    print(f"softmax  torch: {ms_torch:.3f} ms ({gbps(ms_torch):.0f} GB/s)   triton: {ms_triton:.3f} ms ({gbps(ms_triton):.0f} GB/s)")

    a = torch.randn(2048, 2048, device="cuda", dtype=torch.float16)
    b = torch.randn(2048, 2048, device="cuda", dtype=torch.float16)
    tflops = lambda ms: 2 * 2048**3 / (ms * 1e-3) / 1e12
    ms_torch = do_bench(lambda: a @ b)
    ms_triton = do_bench(lambda: triton_matmul(a, b))
    print(f"matmul   torch: {tflops(ms_torch):.1f} TFLOP/s   triton (untuned): {tflops(ms_triton):.1f} TFLOP/s")

The untuned matmul is usually slower than cuBLAS. Next steps to close the gap, in order of payoff:
1. `@triton.autotune` over a few `(BLOCK_M, BLOCK_N, BLOCK_K, num_warps, num_stages)` configs.
2. Grouped ordering of program ids so neighbouring programs reuse the same A/B tiles in L2
   (see the official Triton matmul tutorial).
3. On Hopper and newer GPUs, tensor-memory-accelerator (TMA) descriptors.

## Things to be able to explain

- What does one Triton program correspond to on the hardware? (A thread block / CUDA "CTA"; on AMD, a workgroup.)
- Why must `BLOCK` be a power of two, and what happens to the lanes past `n_cols`?
- `num_warps` and `num_stages`: what do they trade off? (Parallelism within a block; software pipelining of loads
  vs shared-memory use.)

## Drill

Write the kernel *and* its launcher. Your function takes torch tensors and returns a torch tensor.

In [ ]:
def my_triton_add(x, y):
    raise NotImplementedError


def my_triton_softmax(x):
    raise NotImplementedError


if HAS_GPU:
    drill(test_add, my_triton_add)
    drill(test_softmax, my_triton_softmax)